# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and strictly referencing dataset entities by their `@id` as specified in the Croissant schema. The workflow covers loading the Croissant schema, identifying entities by `@id`, data extraction, common preprocessing, and visualization.

### Dataset Source
Source Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL from FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Keywords:", getattr(metadata, 'keywords', []))
print("License:", metadata.license)
print("Citation As:", getattr(metadata, 'citeAs', ''))


## 2. Data Overview
Review all available record sets with their unique `@id` values and enumerate their fields. All references are by `@id` in line with best practices.

In [ ]:
# Retrieve all record set `@id`s
record_sets_metadata = getattr(metadata, 'recordSet', [])
if isinstance(record_sets_metadata, dict):
    record_sets_metadata = [record_sets_metadata]

record_set_ids = []
if record_sets_metadata:
    for rs in record_sets_metadata:
        # The @id can be nested as either a string or an object w/ @id
        rsid = rs["@id"] if isinstance(rs, dict) and "@id" in rs else rs
        print(f"RecordSet @id: {rsid}")
        record_set_ids.append(rsid)
        # Try to print Fields in the RecordSet
        # This relies on the mlcroissant schema reader; if available, get info
        rs_obj = None
        for obj in getattr(metadata, '_jsonld', []):
            if isinstance(obj, dict) and obj.get("@id") == rsid:
                rs_obj = obj
                break
        if rs_obj and 'field' in rs_obj:
            fields = rs_obj['field']
            if isinstance(fields, dict):
                fields = [fields]
            print(f"  Fields in {rsid}:")
            for f in fields:
                field_id = f["@id"] if isinstance(f, dict) and '@id' in f else f
                print(f"    - Field @id: {field_id}")
else:
    # If mlcroissant doesn't parse .recordSet, try to extract record sets from the schema
    print("No explicit record sets available in metadata; attempting to auto-detect via dataset.records().")
    # Try inferring by calling .records() with no argument; often, it returns the main table
    try:
        sample = dataset.records(record_set=None)
        print("Main record set is available (use record_set=None).")
        record_set_ids = [None]
    except Exception as e:
        print(f"Could not auto-detect record sets: {e}")
        record_set_ids = []


## 3. Data Extraction
Load data from all available record sets (referenced by their `@id`) into DataFrames for further analysis. Print all column names for the primary record set.

In [ ]:
# If there are no explicit record sets, default to main data table (record_set=None)
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for RecordSet @id: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Columns for RecordSet @id {rsid}:\n", df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Failed to extract for {rsid}: {e}")


## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field within the main record set:
- Filter by value
- Normalize
- Grouping

**All columns/fields below are referenced using their `@id` as given in the Croissant schema, not by short name.**


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# Pick primary record set (or None if only one, as above)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id].copy()

# Try to discover likely numeric fields by dtype
numeric_field_ids = [col for col in df.columns if np.issubdtype(df[col].dropna().values.dtype, np.number)]
if not numeric_field_ids:
    # Try to infer field with name matching common numeric fields
    likely_candidates = [col for col in df.columns if col.lower().endswith(('mw', 'weight', 'abundance', 'count', 'coverage', 'pi', 'score'))]
    if likely_candidates:
        try_numeric_field = likely_candidates[0]
        df[try_numeric_field] = pd.to_numeric(df[try_numeric_field], errors='coerce')
        numeric_field_ids = [try_numeric_field]

if numeric_field_ids:
    numeric_field = numeric_field_ids[0]  # Use the first numeric field found
    print(f"Using numeric field @id: {numeric_field}")

    # Filter records based on a threshold (choose 10, may be adjusted)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:\n", filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field (e.g., accession, protein name, or sample group)
    group_candidates = [c for c in df.columns if ('accession' in c.lower()) or ('group' in c.lower()) or ('sample' in c.lower()) or ('class' in c.lower())]
    group_field = group_candidates[0] if group_candidates else None
    if group_field is not None and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean) by {group_field} for filtered data:")
        print(grouped_df.head())
else:
    print("No numeric fields discovered for EDA.")


## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized values, as well as group means if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.xlabel(f"{numeric_field} (normalized)")
        plt.title(f"Normalized Distribution ({numeric_field})")
        plt.show()

    # Scatter plot by group if available
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} across groups ({group_field})")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset Mass Spectrometry Analysis of Extracellular Vesicles using `mlcroissant`, referencing entities by their Croissant schema `@id`. We:
- Discovered available record sets and fields by their `@id`
- Loaded data for analysis
- Applied common EDA operations: filtering, normalization, and grouping
- Visualized numeric field distributions and group means

Continue exploring the dataset using the provided `@id`-based references for robust, standards-based data science workflows.
